## Data Curation para estandarización entre datasets y downstream tasks

In [23]:
import pandas as pd
import re
from pathlib import Path

# Rutas a tus archivos (ajusta si hace falta)
bcn_path = Path("../datasets_metadata/csv/2_metadata_BCNB.csv")
hsi_path = Path("../datasets_metadata/xlsx/7_metadata_HSI-BRCA.xlsx")
histai_path = Path("../datasets_metadata/csv/histai_breast_metadata_regroup.csv")

bcn = pd.read_csv(bcn_path)
hsi = pd.read_excel(hsi_path, sheet_name=1)
histai = pd.read_csv(histai_path)

In [24]:

print(hsi.columns.tolist())

['Project Short Name', 'Case ID', 'Race', 'Ethnicity', 'Sex at Birth', 'Age at Diagnosis (Years)', 'Menopausal_status', 'Dx_surgery', 'Tumor_diameter', 'Tumor_histologic_grade', 'LVI', 'PNI', 'T', 'N', 'M', 'ER', 'PR', 'HER2', 'KI67', 'Molecular_subtype', 'LN_status', 'LN_ITC_number', 'LN_MICRO_number', 'LN_MACRO_number', 'LN_number', 'SLN_number', 'SLN_status', 'Tx_hormonal', 'Tx_CT', 'Tx_trastuzumab', 'Tx_RT', 'Relapse', 'Metastasis_type', 'DFS', 'Vital_status', 'Death_cause', 'OS']


### BCNB


Normalizar ER / PR / HER2

In [25]:
def binarize_pos_neg(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip().lower()
    if x.startswith("pos"):
        return 1
    if x.startswith("neg"):
        return 0
    return pd.NA

def map_her2_expr(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip()
    # normalizamos
    if x in ["0", "0+", "score 0"]:
        return 0  # negativo
    if x in ["1", "1+"]:
        return 0  # negativo (ASCO/CAP)
    if x in ["2", "2+"]:
        return 1  # equívoco
    if x in ["3", "3+"]:
        return 2  # positivo
    return pd.NA

bcn["ER_status_norm"] = bcn["ER"].apply(binarize_pos_neg)
bcn["PR_status_norm"] = bcn["PR"].apply(binarize_pos_neg)
bcn["HER2_status_norm"] = bcn["HER2 Expression"].apply(map_her2_expr)

Normalizar Ki‑67 (BCNB)

In [26]:
import numpy as np

def parse_ki67(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().replace("%", "")
    # rango tipo "15-20"
    if "-" in s:
        parts = s.split("-")
        try:
            nums = [float(p) for p in parts if p]
            return np.mean(nums)
        except ValueError:
            return np.nan
    # valor único
    try:
        return float(s)
    except ValueError:
        return np.nan

bcn["Ki67_percent"] = bcn["Ki67"].apply(parse_ki67)

def cat_ki67(p):
    if pd.isna(p):
        return pd.NA
    if p < 10:
        return "low"
    if p <= 20:
        return "intermediate"
    return "high"

bcn["Ki67_cat"] = bcn["Ki67_percent"].apply(cat_ki67)

Normalizar N‑status y tamaño

In [27]:
def map_aln_status(row):
    # ejemplo con BCNB: usa ALN status + número de metástasis
    npos = row["Number of lymph node metastases"]
    if pd.isna(npos):
        return pd.NA
    try:
        npos = int(npos)
    except ValueError:
        return pd.NA

    if npos == 0:
        return "N0"
    if 1 <= npos <= 2:
        return "N+(1-2)"
    if npos > 2:
        return "N+(>2)"
    return pd.NA

bcn["N_status_norm"] = bcn.apply(map_aln_status, axis=1)

def cat_tumor_size_cm(x):
    if pd.isna(x):
        return pd.NA
    try:
        val = float(x)
    except ValueError:
        return pd.NA
    if val <= 2:
        return "T1_≤2cm"
    elif val <= 5:
        return "T2_>2-≤5cm"
    else:
        return "T3+_>5cm"

bcn["Tumor_size_cat"] = bcn["Tumour Size(cm)"].apply(cat_tumor_size_cm)

### HSI

Normalizar ER, PR y HER2

In [28]:
def map_er_pr_code(x):
    if pd.isna(x):
        return pd.NA
    try:
        x = int(x)
    except ValueError:
        return pd.NA
    if x == 0:
        return 0  # negativo
    if x == 1:
        return 1  # positivo
    # otros códigos (2, 3, 9, etc.) los tratamos como missing/indeterminado
    return pd.NA

def map_her2_code(x):
    if pd.isna(x):
        return pd.NA
    try:
        x = int(x)
    except ValueError:
        return pd.NA
    # ejemplo típico:
    # 0 = negativo, 1 = positivo, 2 = equívoco/indeterminado (ajusta según diccionario real)
    if x == 0:
        return 0   # HER2-
    if x == 1:
        return 2   # HER2+ (equivalente a 3+ en score)
    if x == 2:
        return 1   # equívoco
    return pd.NA

hsi["ER_status_norm"] = hsi["ER"].apply(map_er_pr_code)
hsi["PR_status_norm"] = hsi["PR"].apply(map_er_pr_code)
hsi["HER2_status_norm"] = hsi["HER2"].apply(map_her2_code)

Normalizar Ki67

In [29]:
def map_ki67_code_to_percent(x):
    if pd.isna(x):
        return np.nan
    try:
        code = int(x)
    except ValueError:
        return np.nan

    # EJEMPLO: mapeo heurístico; ajusta a tu codificación real
    mapping = {
        0: 5.0,   # muy bajo
        1: 10.0,  # bajo
        2: 20.0,  # intermedio
        3: 30.0,  # alto
        4: 40.0,  # muy alto
    }
    return mapping.get(code, np.nan)

hsi["Ki67_percent"] = hsi["KI67"].apply(map_ki67_code_to_percent)

def cat_ki67(p):
    if pd.isna(p):
        return pd.NA
    if p < 10:
        return "low"
    if p <= 20:
        return "intermediate"
    return "high"

hsi["Ki67_cat"] = hsi["Ki67_percent"].apply(cat_ki67)

Tamaño tumoral

In [30]:
def cat_tumor_size_mm(x):
    if pd.isna(x):
        return pd.NA
    try:
        val = float(x)
    except ValueError:
        return pd.NA

    # convertimos a cm
    cm = val / 10.0
    if cm <= 2:
        return "T1_≤2cm"
    elif cm <= 5:
        return "T2_>2-≤5cm"
    else:
        return "T3+_>5cm"

hsi["Tumor_size_cat"] = hsi["Tumor_diameter"].apply(cat_tumor_size_mm)

N‑status a partir de N y de LN_*

In [31]:
def map_n_status(row):
    # Usamos primero conteo de ganglios positivos si está disponible
    macro = row.get("LN_MACRO_number", np.nan)
    micro = row.get("LN_MICRO_number", np.nan)
    itc = row.get("LN_ITC_number", np.nan)

    pos_total = 0
    for v in [macro, micro, itc]:
        if pd.notna(v):
            try:
                pos_total += int(v)
            except ValueError:
                pass

    if pos_total > 0:
        if pos_total <= 2:
            return "N+(1-2)"
        else:
            return "N+(>2)"

    # Si no tenemos conteos o son 0, miramos N directamente
    n_code = row.get("N", pd.NA)
    if pd.isna(n_code):
        return pd.NA
    try:
        n_code = int(n_code)
    except ValueError:
        return pd.NA

    if n_code == 0:
        return "N0"
    if n_code == 1:
        return "N+(1-2)"   # aproximación
    if n_code >= 2:
        return "N+(>2)"
    return pd.NA

hsi["N_status_norm"] = hsi.apply(map_n_status, axis=1)

## Extracción y reagrupación en HISTAI (regex)

In [32]:
histai_text_cols = ["conclusion", "additional_info", "micro_protocol"]

def concat_text(row):
    parts = []
    for c in histai_text_cols:
        if c in row and pd.notna(row[c]):
            parts.append(str(row[c]))
    return " ".join(parts)

histai["full_text"] = histai.apply(concat_text, axis=1)

ER/PR:

In [33]:
def extract_er_status(text):
    text_low = text.lower()
    # busca "er positive" / "er negative"
    if re.search(r"\ber\b[^.]*negative", text_low):
        return 0
    if re.search(r"\ber\b[^.]*positive", text_low):
        return 1
    return pd.NA

def extract_pr_status(text):
    text_low = text.lower()
    if re.search(r"\bpr\b[^.]*negative", text_low):
        return 0
    if re.search(r"\bpr\b[^.]*positive", text_low):
        return 1
    return pd.NA

histai["ER_status_norm"] = histai["full_text"].fillna("").apply(extract_er_status)
histai["PR_status_norm"] = histai["full_text"].fillna("").apply(extract_pr_status)

HER2:

In [34]:
def extract_her2_score(text):
    text_low = text.lower()
    # busca patrones tipo "her2 0", "her2 1+", "her2 2+", "her2 3+"
    m = re.search(r"her2[^0-9]*([0-3])\s*\+?", text_low)
    if m:
        try:
            score = int(m.group(1))
        except ValueError:
            score = None
        if score is not None:
            if score in [0, 1]:
                return 0
            if score == 2:
                return 1
            if score == 3:
                return 2
    # si no hay número, busca negative/positive
    if "her2" in text_low and "negative" in text_low:
        return 0
    # casos con "undetermined", "equivocal"
    if "her2" in text_low and ("undetermined" in text_low or "equivocal" in text_low):
        return 1
    return pd.NA

histai["HER2_status_norm"] = histai["full_text"].fillna("").apply(extract_her2_score)

Ki‑67:

In [35]:
def extract_ki67(text):
    # busca "Ki-67 15" o "Ki-67 39.8"
    m = re.search(r"ki-?67[^0-9]*([0-9]+(?:\.[0-9]+)?)", text.lower())
    if m:
        try:
            return float(m.group(1))
        except ValueError:
            return np.nan
    return np.nan

histai["Ki67_percent"] = histai["full_text"].fillna("").apply(extract_ki67)
histai["Ki67_cat"] = histai["Ki67_percent"].apply(cat_ki67)  # reutilizamos cat_ki67 de antes

Subtipo molecular a partir de texto

In [36]:
def extract_subtype(text):
    t = text.lower()
    if "triple-negative" in t or "triple negative" in t:
        return "Triple negative"
    if "luminal a" in t:
        return "Luminal A"
    if "luminal b" in t:
        return "Luminal B"
    if "her2" in t and ("positive" in t or "3+" in t):
        return "HER2+"
    return pd.NA

histai["Molecular_subtype_norm"] = histai["full_text"].fillna("").apply(extract_subtype)

In [37]:
from pathlib import Path

output_dir = Path("output_norm")
output_dir.mkdir(exist_ok=True)

# Versión extendida con columnas nuevas, pero manteniendo todo lo original
bcn.to_csv(output_dir / "2_metadata_BCNB_curated.csv", index=False)
hsi.to_excel(output_dir / "7_metadata_HSI-BRCA-2_curated.xlsx", index=False)
histai.to_csv(output_dir / "histai_breast_metadata_regroup-3_curated.csv", index=False)